# Task 5 – IB9AU – 2026

**Name:** qizhou lin  
**Task:** Semantic Search for Financial News with Sentence Embeddings and Gradio

**Key Insights:**
- Learned how to clean and split text columns to extract URLs.
- Generated sentence embeddings using a pre-trained transformer.
- Built a semantic search app with Gradio using cosine similarity.
- Practiced combining NLP with UI tools for interactive search functionality.


In [2]:
from google.colab import files
uploaded = files.upload()


Saving financial_news.csv to financial_news.csv


In [3]:
df = pd.read_csv("financial_news.csv")
df.head()


,text,label
0,Here are Thursday's biggest analyst calls: App...,0
1,Buy Las Vegas Sands as travel to Singapore bui...,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",0
3,"Analysts react to Tesla's latest earnings, bre...",0
4,Netflix and its peers are set for a ‘return to...,0


In [4]:
import re

df["URL"] = df["text"].apply(lambda x: re.findall(r'(https?://\S+)', x)[-1] if re.findall(r'(https?://\S+)', x) else "")
df["text"] = df["text"].apply(lambda x: re.sub(r'(https?://\S+)', '', x).strip())
df.head()


,text,label,URL
0,Here are Thursday's biggest analyst calls: App...,0,https://t.co/QPN8Gwl7Uh
1,Buy Las Vegas Sands as travel to Singapore bui...,0,https://t.co/fLS2w57iCz
2,"Piper Sandler downgrades DocuSign to sell, cit...",0,https://t.co/1EmtywmYpr
3,"Analysts react to Tesla's latest earnings, bre...",0,https://t.co/kwhoE6W06u
4,Netflix and its peers are set for a ‘return to...,0,https://t.co/jPpdl0D9s4


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['text'].tolist(), convert_to_tensor=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
import gradio as gr
import torch
from sklearn.metrics.pairwise import cosine_similarity

def search_news(query):
    query_embedding = model.encode([query], convert_to_tensor=True)
    similarities = cosine_similarity(query_embedding.cpu().numpy(), embeddings.cpu().numpy())[0]
    top_indices = similarities.argsort()[-5:][::-1]
    results = df.iloc[top_indices][['text', 'URL']]
    return results.to_dict(orient='records')


In [7]:
iface = gr.Interface(
    fn=search_news,
    inputs=gr.Textbox(lines=2, placeholder="Enter a search query..."),
    outputs="json",
    title="Financial News Semantic Search",
    description="Enter a phrase (e.g. 'regulatory fine', 'earnings surprise') to find the top 5 most similar news headlines."
)

iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dce5c345a95fe17798.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
